# 長い文書をチャンクに分割する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/16_llm/chunking.ipynb)

NLP クックブック 16-08 ／ 大規模言語モデル (LLM)

RAG のために長い文書を検索しやすい大きさの断片に区切りたい。文の途中で切らず、前後を少し重ねたい。

In [1]:
import io, re, urllib.request, zipfile

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

text = aozora("https://www.aozora.gr.jp/cards/000035/files/1567_ruby_4948.zip")   # 走れメロス
print(len(text), "文字")

def chunk_by_sentences(text, max_chars=400, overlap_sents=1):
    """文の境界を保ったまま、max_chars を超えないように詰める。隣のチャンクと overlap_sents 文だけ重ねる"""
    sents = [s.strip() for s in re.split(r"(?<=[。！？」])(?![」。！？])|\n+", text) if s and s.strip()]
    chunks, cur = [], []
    for s in sents:
        if cur and sum(map(len, cur)) + len(s) > max_chars:
            chunks.append("".join(cur))
            cur = cur[-overlap_sents:] if overlap_sents else []
        cur.append(s)
    if cur:
        chunks.append("".join(cur))
    return chunks

chunks = chunk_by_sentences(text)
print(len(chunks), "チャンク / 文字数の最小・平均・最大:", min(map(len, chunks)), sum(map(len, chunks)) // len(chunks), max(map(len, chunks)))
for c in chunks[:2]:
    print("---", c[:150], "...")

9913 文字
27 チャンク / 文字数の最小・平均・最大: 269 383 400
--- メロスは激怒した。必ず、かの邪智暴虐の王を除かなければならぬと決意した。メロスには政治がわからぬ。メロスは、村の牧人である。笛を吹き、羊と遊んで暮して来た。けれども邪悪に対しては、人一倍に敏感であった。きょう未明メロスは村を出発し、野を越え山越え、十里はなれた此のシラクスの市にやって来た。メロスには ...
--- 久しく逢わなかったのだから、訪ねて行くのが楽しみである。歩いているうちにメロスは、まちの様子を怪しく思った。ひっそりしている。もう既に日も落ちて、まちの暗いのは当りまえだが、けれども、なんだか、夜のせいばかりでは無く、市全体が、やけに寂しい。のんきなメロスも、だんだん不安になって来た。路で逢った若い ...


In [2]:
# トークン数を基準にする場合 (埋め込みモデルの最大長に合わせる)
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")

def chunk_by_tokens(text, max_tokens=256, overlap=32):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return [tok.decode(ids[i:i + max_tokens]) for i in range(0, len(ids), max_tokens - overlap)]

tchunks = chunk_by_tokens(text)
print(len(tchunks), "チャンク", [len(tok(c)["input_ids"]) for c in tchunks[:5]])

31 チャンク [258, 259, 258, 259, 260]


- チャンクが小さすぎると文脈が失われ、大きすぎると検索の精度が落ちて LLM に渡すトークンも増える。200〜500 トークン程度から試す。
- 構造のある文書 (Markdown、論文、マニュアル) は、見出しや段落を単位にするのが最も良い。各チャンクに文書タイトルや見出しを付け足すと検索精度が上がる。
- トークン数だけで機械的に切ると文の途中で切れる。両方を組み合わせる (文で区切り、トークン数で上限を設ける) のが実用的。